In [4]:
import pandas as pd
from src.config import *
from src.utils import *
from src.predictor.utils import build_prediction_rows  # fonction définie juste avant
from datetime import datetime

# ⏱️ Suivi de l'exécution
start_time = datetime.now()
print("Start time:", start_time)

# 🔁 Charger le dataset complet
dataset_path = get_latest_file(DATA_FINAL_CLEANED_DATASET_DIR)
full_df = pd.read_csv(dataset_path)
print("Loaded dataset:", dataset_path)

# ✅ Vérifications de base
print("Full dataset shape:", full_df.shape)
assert "IS_WIN" in full_df.columns, "La colonne IS_WIN doit être présente dans le dataset"

# 🧪 Extraction des deux dernières lignes (réelles)
real_last_rows = full_df.tail(2).copy()
home_id = real_last_rows.loc[real_last_rows["IS_HOME"] == 1, "TEAM_ID"].values[0]
away_id = real_last_rows.loc[real_last_rows["IS_HOME"] == 0, "TEAM_ID"].values[0]
print(f"Match: HOME={home_id}, AWAY={away_id}")

# 🧼 Dataset sans ces lignes
cut_df = full_df.iloc[:-2].copy()

# 🔁 Reconstruction des lignes à prédire
predicted_rows = build_prediction_rows(home_team_id=home_id, away_team_id=away_id, dataset=cut_df)

# 🧹 Préparation des lignes réelles pour comparaison (mêmes colonnes, sans target ni infos post-match)
drop_cols = ['TEAM_ID','OPP_TEAM_ID','SEASON','GAME_DATE','IS_WIN']
real_clean = real_last_rows.drop(columns=drop_cols, errors='ignore').reset_index(drop=True)
pred_clean = predicted_rows.reset_index(drop=True)


# Assure une structure identique
common_cols = [col for col in real_clean.columns if col in pred_clean.columns]
real_clean = real_clean[common_cols]
pred_clean = pred_clean[common_cols]

# Réindexer pour être sûrs
real_clean = real_clean.reset_index(drop=True)
pred_clean = pred_clean.reset_index(drop=True)


# 🧪 Comparaison
diff = (real_clean != pred_clean).sum()
diff_summary = diff[diff > 0]

print(f"\nNombre de colonnes différentes : {len(diff_summary)}")
print("Colonnes divergentes :", diff_summary.index.tolist())

# 🔍 Affichage côte à côte des lignes si besoin
pd.set_option("display.max_columns", None)
print("\nLigne prédite:")
display(pred_clean)
print("\nLigne réelle:")
display(real_clean)

# 🕒 Fin
end_time = datetime.now()
print("End time:", end_time)
print("Duration:", end_time - start_time)


Start time: 2025-05-25 17:27:58.013929
Loaded dataset: data/final_cleaned_dataset/nba_features_cleaned_final_2025-05-25_16-22-56.csv
Full dataset shape: (64146, 188)
Match: HOME=1610612750, AWAY=1610612760


KeyError: 'Column not found: PTS'